# 🩺💉💊 <span style="color: white; background-color: tomato"><b> Extração do Relatório de Atestados Médicos </b></span></p>
 
🔧 <span style="color: chocolate"><b> 1- Acessa automaticamente o Datamace (MO) </b></span></p>
- O script verifica se a janela do Datamace (HTML5) já está aberta
- Caso não esteja:
    - Abre o navegador  
    - Realiza o login Cloud via Selenium  
    - Aguarda carregamento  
    - Realiza o segundo login via PyAutoGUI
- Se a janela já estiver ativa, apenas a ativa e preenche o login diretamente

🩺 <span style="color: chocolate"><b> 2- Navega pelo módulo Medicina Ocupacional </b></span></p>
- Via PyAutoGUI, o script segue exatamente o fluxo do MO:
    - Acessa o módulo Medicina Ocupacional  
    - Entra em Ambulatório → Atestado Médico  
    - Abre o relatório de Atestado Médico  
    - Define opções de multiprocessamento (Situação, Observação, Saída)  
    - Seleciona Excel como método de exportação  
    - Confirma a extração
- É essencialmente a reprodução automatizada do comportamento humano na interface gráfica

⏳ <span style="color: chocolate"><b> 3- Aguarda confirmação do download </b></span></p>
- Exibe uma caixa PyAutoGUI perguntando:
    - “Foi realizado o download da base ATESTADOS?”
- Essa etapa garante que o arquivo realmente chegou na pasta Downloads antes de iniciar o processamento

🧹 <span style="color: chocolate"><b> 4- Processa a base ATESTADOS </b></span></p>
- O script realiza uma limpeza e padronização completa:
    - Conversão de datas: Colunas como Data, Data Ini., Data Final, etc., são convertidas para datetime
    - Padronização da coluna Registro: Registros são truncados para os 6 primeiros caracteres + limpeza de espaços
    - Conversão do campo "Tempo": Converte string para número (transformando vírgula em ponto)
    - Mapeamento e renomeação de colunas: Usa um dicionário MAPEAMENTO_COLUNAS para transformar todas as colunas originais do Datamace em nomes padronizados e consistentes com suas outras bases corporativas
- Limpeza de valores nulos:
    - Converte NaN → None (compatível com Excel)
- Logs detalhados:
    - Registra no workbook de PROCESSOS.xlsx todas as etapas da execução

📁 <span style="color: chocolate"><b> 5- Busca e consolida todos os arquivos exportados </b></span></p>
- Busca arquivos iniciados com o padrão definido (RFA13C)  
- Processa cada arquivo  
- Move os arquivos já tratados para a pasta de arquivos processados  
- Concatena todas as bases  
- Elimina duplicidades

📊 <span style="color: chocolate"><b> 6- Gera o Excel consolidado formatado </b></span></p>
- Cria o arquivo ATESTADOS.xlsx com:
    - Sheet ATESTADOS
    - Tabela Excel formatada com TableStyleLight13
    - Colunas padronizadas
    - Dados limpos e estruturados

🔄 <span style="color: chocolate"><b> 7- Atualiza automaticamente o arquivo corporativo Controle HC e Atestados </b></span></p>
- Abre o arquivo, navega até a aba ATESTADOS (célula B5), e dispara o comando de refresh (ALT + F5)
- Atualiza automaticamente os dados consumidos pelo RH.

🗃 <span style="color: chocolate"><b> 8- Exporta a base final para CSV </b></span></p>
- Gera tb_atestados.csv
- Pronto para ingestão no banco de dados ou pipelines de ETL.

📝 <span style="color: chocolate"><b> 9- Gera um resumo e calcula o tempo total de execução </b></span></p>
- No final, exibe:
    - Confirmação de processo finalizado  
    - Tempo completo de execução

# Importando as Bibliotecas

In [1]:
import logging
import os
import pandas as pd
import pyautogui
import pygetwindow as gw
import pyperclip
import shutil
import socket
import sys
import time
import tkinter as tk
import win32com.client
import win32gui, win32con
from asyncio.log import logger
from contextlib import contextmanager
from datetime import datetime, date
from dotenv import load_dotenv
from openpyxl import load_workbook, Workbook
from openpyxl.utils.dataframe import dataframe_to_rows
from openpyxl.worksheet.table import Table, TableStyleInfo
from pathlib import Path
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver import ActionChains
from tkinter import simpledialog
from webdriver_manager.chrome import ChromeDriverManager
from selenium.common.exceptions import WebDriverException

tempo_0 = [id, datetime.today(), 0]

print('----------------------------------------------------------------------------------------------------')
print('')
print('   ✅ Processo de Importação finalizado')
print('')
print('----------------------------------------------------------------------------------------------------')

----------------------------------------------------------------------------------------------------

   ✅ Processo de Importação finalizado

----------------------------------------------------------------------------------------------------


# Caminhos de Pastas

In [2]:
# Configuração
CONFIG = {
    'id_processo': 4,
    'processos': Path(r'X:\Gestão de Pessoas\Analytics\03 - Bases\1. BASES TRATADAS\PROCESSOS.xlsx'),
    'origem': Path(r'C:\Users\rodrigo.bernandes\Downloads'),
    'movidos': Path(r'X:\Gestão de Pessoas\Analytics\03 - Bases\2. ARQUIVOS MOVIDOS'),
    'saida': Path(r'X:\Gestão de Pessoas\Analytics\03 - Bases\1. BASES TRATADAS\ATESTADOS.xlsx'),
    'env_path': Path(r'X:\Gestão de Pessoas\Analytics\08 - Notebooks Python\08.3 - Automações\.env'),
    'padrao': 'RFA13C',
}

# Registra Etapa do Processo

In [3]:
def append_registro_processo(caminho, id_proc, etapa):
    try:
        wb = load_workbook(caminho)
        ws = wb['REGISTROS']
        ws.append([id_proc, datetime.today(), etapa])
        wb.save(caminho)
        wb.close()
    except Exception as e:
        print(f"❌ Erro ao registrar etapa {etapa}: {e}")

print('----------------------------------------------------------------------------------------------------')
print('')
print('   ✅ Registro de processos')
print('')
print('----------------------------------------------------------------------------------------------------')

----------------------------------------------------------------------------------------------------

   ✅ Registro de processos

----------------------------------------------------------------------------------------------------


# Configurações Iniciais e Acessando o Datamace

In [4]:
load_dotenv(dotenv_path='env_path')

cloud_user = os.getenv("CLOUD_USER")
cloud_password = os.getenv("CLOUD_PASSWORD")
app_user = os.getenv("APP_USER")
app_password = os.getenv("APP_PASSWORD")
datamace = os.getenv("SITE_DATAMACE")

# Validar se todas existem
if not all([cloud_user, cloud_password, app_user, app_password]):
    raise ValueError("Uma ou mais variáveis de ambiente não foram encontradas. Verifique o arquivo .env.")

titulo_aba = "HTML5"

# Verifica se janela 'HTML5' está aberta
if gw.getWindowsWithTitle(titulo_aba):
    pyautogui.click(836, 445)
    time.sleep(0.5)
    pyautogui.typewrite(app_user)
    pyautogui.press('tab')
    time.sleep(0.5)
    pyautogui.typewrite(app_password)
    pyautogui.press('enter')
else:
    driver = webdriver.Chrome()
    time.sleep(1)
    window = win32gui.GetForegroundWindow()
    win32gui.ShowWindow(window, win32con.SW_MAXIMIZE)
    time.sleep(1)
    driver.get(datamace)
    time.sleep(8)
    # Login cloud
    driver.find_element(By.NAME, "username").send_keys(cloud_user)
    driver.find_element(By.NAME, "Password").send_keys(cloud_password)
    driver.find_element(By.NAME, "Password").send_keys(Keys.ENTER)
    time.sleep(30)
    # Login pyautogui
    pyautogui.click(836, 445)
    time.sleep(0.5)
    pyautogui.typewrite(app_user)
    pyautogui.press('tab')
    time.sleep(0.5)
    pyautogui.typewrite(app_password)
    pyautogui.press('enter')

print("🏁 Acesso finalizado")

🏁 Acesso finalizado


# Acessando o MO

In [5]:
# Fechando janelas de Novidades do Datamace
time.sleep(3)
pyautogui.press('enter')
time.sleep(3)
pyautogui.click(x=1268, y=199)
# Acessando o MO
time.sleep(10) # Tempo maior para aguardar carregar o MO
pyautogui.click(x=474, y=196) # Clicando no módulo MO
time.sleep(3)
pyautogui.click(x=1079, y=235) # Fecha a janela de Tarefas Anuais
time.sleep(2)
pyautogui.click(x=52, y=299) # Ambulatório
time.sleep(1)
pyautogui.click(x=147, y=590) # Atestado Médico
time.sleep(1)
pyautogui.click(x=313, y=627) # Relatório Atestado Médico
time.sleep(2)
pyautogui.click(x=531, y=283) # Multi processamento
time.sleep(1)
pyautogui.click(x=862, y=599) # Situação
time.sleep(1)
pyautogui.click(x=850, y=642) # Todos
time.sleep(1)
pyautogui.click(x=776, y=645) # Observação
time.sleep(1)
pyautogui.click(x=655, y=654) # Saída
time.sleep(2)
pyautogui.click(x=655, y=682) # Excel
time.sleep(2)
pyautogui.click(x=557, y=717) # Confirmar
time.sleep(2)

print('----------------------------------------------------------------------------------------------------')
print('')
print('   ✅ Extraindo relatório')
print('')
print('----------------------------------------------------------------------------------------------------')

----------------------------------------------------------------------------------------------------

   ✅ Extraindo relatório

----------------------------------------------------------------------------------------------------


# Aguardando a Conclusão do Download da Base ATESTADOS

In [6]:
# Configuração de logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")

def verificar_download_base() -> bool:
    """
    Exibe uma caixa de diálogo para confirmar se o download da base COLAB foi realizado.
    """
    tipo_escolhido = pyautogui.confirm(
        text='Foi realizado o download da base ATESTADOS?', 
        title='Seleção de Extração', 
        buttons=['Sim']
    )

    # Se fechar a janela
    if tipo_escolhido is None:
        logging.warning("Operação cancelada pelo usuário no prompt inicial. ❌ Encerrando o script.")
        sys.exit(0)
        
    # Se for 'Sim'
    logging.info(f"Opção selecionada: {tipo_escolhido}. ✅ Continuando o processo...")
    return True

# --- Execução principal ---
if __name__ == "__main__":
    # Chama a função de validação antes de seguir com o restante do código
    verificar_download_base()
    
    logging.info("Iniciando a leitura e processamento dos dados...")

2026-06-25 10:12:19,526 - INFO - Opção selecionada: Sim. ✅ Continuando o processo...
2026-06-25 10:12:19,594 - INFO - Iniciando a leitura e processamento dos dados...


# Mapeamento de Colunas

In [7]:
MAPEAMENTO_COLUNAS = {
    'Cod.Emp.': 'cod_empresa',
    'Registro': 'registro',
    'Nome Trab.': 'nome',
    'CPF': 'cpf',
    'Data Nasc.': 'nascimento',
    'Sexo': 'sexo',
    'Data': 'data_atestado',
    'Tab.Cons.': 'cod_consultorio',
    'Nome Cons.': 'consultorio',
    'Tab.Med.': 'cod_medico',
    'Nome Med.': 'medico',
    'Conselho': 'conselho',
    'Num.Cons.': 'registro_conselho',
    'Data Ini.': 'data_inicio',
    'Data Final': 'data_fim',
    'Tempo': 'tempo',
    'Tipo Tem.': 'tipo_tempo',
    'Hor. entr.': 'hora_entrada',
    'Hor. saída': 'hora_saida',
    'CID': 'cid',
    'Desc.CID': 'descricao_cid',
    'Capitulo': 'capitulo',
    'Grupo Ini.': 'grupo_inicial',
    'Grupo Fin.': 'grupo_final',
    'Categoria': 'categoria',
    'NTEP': 'ntep',
    'Aceito': 'aceito',
    'Motivo': 'motivo',
    'Observacao': 'observacao',
    'Des.Cargo': 'cargo_abreviado',
    'Data inc.': 'data_inclusao',
    'Usu.incl.': 'usuario_inclusao',
    'Data alt.': 'data_alteracao',
    'Usu.alte.': 'usuario_alteracao',
    'Hr.não.tr.': 'horas_nao_trabalhadas',
}
COLUNAS_DATA = ['Data Nasc.', 'Data', 'Data Ini.', 'Data Final', 'Data inc.', 'Data alt.']

# Processando Atestados Médicos

In [8]:
@contextmanager

def gerenciar_workbook(caminho: Path, sheet: str):
    """Context manager para operações seguras com workbook."""
    wb = load_workbook(caminho)
    ws = wb[sheet]
    try:
        yield ws
    finally:
        wb.save(caminho)

def registrar_etapa(caminho: Path, id_proc: int, etapa: int):
    """Registra etapa do processo."""
    with gerenciar_workbook(caminho, 'REGISTROS') as ws:
        ws.append([id_proc, datetime.today(), etapa])
    logger.debug(f"✅ Etapa {etapa} registrada")

def converter_datas(df: pd.DataFrame, colunas: list) -> pd.DataFrame:
    """Converte colunas para datetime."""
    for col in colunas:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], format='%d/%m/%Y', errors='coerce')
    return df

def limpar_registro(df: pd.DataFrame) -> pd.DataFrame:
    """Limpa coluna REGISTRO (pega apenas primeiros 6 caracteres)."""
    if 'Registro' in df.columns:
        df['Registro'] = df['Registro'].astype(str).str[:6].str.strip()
    return df

def limpar_tempo(df: pd.DataFrame) -> pd.DataFrame:
    """Converte campo Tempo de string para número."""
    if 'Tempo' in df.columns:
        df['Tempo'] = pd.to_numeric(
            df['Tempo'].astype(str).str.replace(',', '.'),
            errors='coerce'
        )
    return df

def mapear_colunas_atestados(df: pd.DataFrame) -> pd.DataFrame:
    """Mapeia colunas do arquivo de atestados."""
    colunas_existentes = {k: v for k, v in MAPEAMENTO_COLUNAS.items() if k in df.columns}
    return df.rename(columns=colunas_existentes)[list(colunas_existentes.values())]

def limpar_valores_nulos(df: pd.DataFrame) -> pd.DataFrame:
    """Limpa valores NaN e converte para None (compatível com Excel)."""
    # Python 3.10+: usar .map() ao invés de .applymap()
    try:
        return df.map(lambda x: None if pd.isna(x) else x)
    except AttributeError:
        # Fallback para versões mais antigas
        return df.applymap(lambda x: None if pd.isna(x) else x)
    
def processar_arquivo(caminho: Path) -> pd.DataFrame:
    """Processa um arquivo de atestados."""
    try:
        logger.debug(f"Carregando: {caminho.name}")
        
        # Carregar
        df = pd.read_excel(caminho, engine='openpyxl')
        
        # Converter tipos
        df = converter_datas(df, COLUNAS_DATA)
        df = limpar_registro(df)
        df = limpar_tempo(df)
        
        # Mapear colunas
        df = mapear_colunas_atestados(df)
        
        # Limpar NaNs
        df = limpar_valores_nulos(df)
        
        logger.debug(f"✅ {len(df)} registros processados")
        return df
        
    except Exception as e:
        logger.error(f"❌ Erro ao processar {caminho.name}: {e}")
        import traceback
        logger.error(traceback.format_exc())
        return None
    
def criar_excel_com_tabela(df: pd.DataFrame, caminho: Path):
    """Cria Excel com tabela formatada."""
    logger.info(f"📝 Criando Excel final ({len(df)} registros)...")
    
    # Salvar com Pandas
    df.to_excel(caminho, sheet_name='ATESTADOS', index=False, engine='openpyxl')
    
    # Aplicar formatação
    wb = load_workbook(caminho)
    ws = wb.active
    
    tabela = Table(displayName="ATESTADOS", ref=ws.dimensions)
    estilo = TableStyleInfo(
        name="TableStyleLight13",
        showFirstColumn=False,
        showLastColumn=False,
        showRowStripes=True,
        showColumnStripes=True
    )
    tabela.tableStyleInfo = estilo
    ws.add_table(tabela)
    
    wb.save(caminho)
    logger.info(f"✅ Excel criado: {caminho}")

# Main
if __name__ == "__main__":
    tempo_inicio = datetime.now()
    
    logger.info("=" * 80)
    logger.info("🔄 PROCESSAMENTO DE ATESTADOS")
    logger.info("=" * 80)

2026-06-25 10:12:19,870 - INFO - ================================================================================
2026-06-25 10:12:19,873 - INFO - 🔄 PROCESSAMENTO DE ATESTADOS
2026-06-25 10:12:19,876 - INFO - ================================================================================


# Processamento e Consolidação do Arquivo

In [9]:
try:
        # Etapa 1: Registrar início
        registrar_etapa(CONFIG['processos'], CONFIG['id_processo'], 0)
            
        # Etapa 2: Buscar arquivos
        logger.info("\n📂 Buscando arquivos...")
        arquivos = sorted([f for f in CONFIG['origem'].iterdir() if f.name.startswith(CONFIG['padrao'])])
            
        if not arquivos:
            logger.warning("❌ Nenhum arquivo encontrado")
            exit()
            
        logger.info(f"✅ {len(arquivos)} arquivo(s) encontrado(s)\n")
            
        registrar_etapa(CONFIG['processos'], CONFIG['id_processo'], 1)
            
        # Etapa 3: Processar arquivos (CONSOLIDAR FORA DO LOOP)
        logger.info("🔄 Processando arquivos...\n")
            
        todas_bases = []
            
        for idx, arquivo in enumerate(arquivos, 1):
            logger.info(f"[{idx}/{len(arquivos)}] {arquivo.name}")
                
            df = processar_arquivo(arquivo)
                
            if df is not None and len(df) > 0:
                todas_bases.append(df)
                logger.info(f"   ✅ {len(df)} registros adicionados")
                    
                try:
                    destino = CONFIG['movidos'] / arquivo.name
                    shutil.move(str(arquivo), str(destino))
                    logger.info(f"   ✅ Movido para arquivos processados\n")
                except Exception as e:
                    logger.error(f"   ⚠️ Erro ao mover: {e}\n")
            else:
                logger.warning(f"   ❌ Sem dados válidos\n")
            
        registrar_etapa(CONFIG['processos'], CONFIG['id_processo'], 2)
            
        # Etapa 4: Consolidar e salvar (UMA ÚNICA VEZ)
        if todas_bases:
            logger.info("💾 Consolidando dados...")
            base_final = pd.concat(todas_bases, ignore_index=True)
            base_final = base_final.drop_duplicates()
            logger.info(f"✅ {len(base_final)} registros consolidados\n")
                
            criar_excel_com_tabela(base_final, CONFIG['saida'])
        else:
            logger.error("❌ Nenhum arquivo foi processado!")
            exit()
            
        # Finalizar
        tempo_duracao = datetime.now() - tempo_inicio
            
        logger.info("\n" + "=" * 80)
        logger.info("✅ PROCESSO FINALIZADO COM SUCESSO!")
        logger.info(f"   Tempo de execução: {tempo_duracao}")
        logger.info("=" * 80)
            
except Exception as e:
    logger.error(f"\n❌ ERRO CRÍTICO: {e}")
    import traceback
    logger.error(traceback.format_exc())

2026-06-25 10:12:20,616 - INFO - 
📂 Buscando arquivos...
2026-06-25 10:12:20,628 - INFO - ✅ 1 arquivo(s) encontrado(s)

2026-06-25 10:12:21,116 - INFO - 🔄 Processando arquivos...

2026-06-25 10:12:21,118 - INFO - [1/1] RFA13C-09321407.XLSX
c:\Users\rodrigo.bernandes\AppData\Local\Programs\Python\Python312\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
2026-06-25 10:12:37,215 - INFO -    ✅ 22313 registros adicionados
2026-06-25 10:12:37,410 - INFO -    ✅ Movido para arquivos processados

2026-06-25 10:12:37,736 - INFO - 💾 Consolidando dados...
2026-06-25 10:12:37,784 - INFO - ✅ 22311 registros consolidados

2026-06-25 10:12:37,784 - INFO - 📝 Criando Excel final (22311 registros)...
2026-06-25 10:13:09,169 - INFO - ✅ Excel criado: X:\Gestão de Pessoas\Analytics\03 - Bases\1. BASES TRATADAS\ATESTADOS.xlsx
2026-06-25 10:13:09,171 - INFO - 
2

# Atualizando o Arquivo Excel Controle HC e Atestados

In [10]:
# Caminho do arquivo
path_excel = r"X:\Gestão de Pessoas\Analytics\10 - Relatórios\10.4 - HC e Atestados Médicos\Controle_HC e Atestados.xlsx"
os.startfile(path_excel) # Abre o arquivo
time.sleep(60)
pyautogui.press('esc')
time.sleep(15)

# Utiliza o comando "Ir para" (Ctrl + G) para navegar até a aba e célula
pyautogui.hotkey('ctrl', 'g')
time.sleep(1)

# Digita o endereço completo
pyautogui.write('ATESTADOS!B5')
time.sleep(1)
pyautogui.press('enter')
time.sleep(3)

#Atualizando o arquivo
pyautogui.hotkey('alt', 'F5')
time.sleep(2)

print('----------------------------------------------------------------------------------------------------')
print('')
print('   ✅ Planilha atualizada com sucesso')
print('')
print('----------------------------------------------------------------------------------------------------')

----------------------------------------------------------------------------------------------------

   ✅ Planilha atualizada com sucesso

----------------------------------------------------------------------------------------------------


# Criando Base em CSV para o Banco de Dados

In [11]:
# Ler o XLSX
df = pd.read_excel(r'X:\Gestão de Pessoas\Analytics\03 - Bases\1. BASES TRATADAS\ATESTADOS.xlsx')

# Salvar como CSV
df.to_csv(r'X:\Gestão de Pessoas\Analytics\03 - Bases\1. BASES TRATADAS\tb_atestados.csv', index=False, encoding='utf-8')

print("✅ Arquivo convertido para CSV com sucesso!")

✅ Arquivo convertido para CSV com sucesso!


# Resumo de Finalização do Processo

In [12]:
tempo_1 = [id, datetime.today(), 1]

print('----------------------------------------------------------------------------------------------------')
print('')
print('     ✅  Processo finalizado')
print('')
print('     ⏱️   Tempo de execução:')
print('')
print(f'   {tempo_1[1] - tempo_0[1]}')
print('')
print('----------------------------------------------------------------------------------------------------')

----------------------------------------------------------------------------------------------------

     ✅  Processo finalizado

     ⏱️   Tempo de execução:

   0:43:04.721950

----------------------------------------------------------------------------------------------------
